# Basic Statistical Concepts

Key measures for understanding data distributions.

**Mean (Average)**
Sum all values, divide by count. Formula: $\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$

Good for symmetric data without extreme outliers. Think of it as the balance point of your data.

**Median (Middle Value)**
The middle value when data is sorted. For even number of values, take average of two middle values.

Better than mean when you have outliers or skewed data. Half your data is above this value, half below.

**Mode (Most Common)**
The value(s) that appear most frequently. 

- Unimodal: one mode
- Bimodal: two modes  
- Multimodal: multiple modes
- No mode: all values equally frequent

Useful for categorical data or finding the "typical" value.

In [71]:
# Quick example with small dataset
x = [2, 4, 4, 4, 5, 5, 7, 9]
print("Data:", x)
print(f"Mean: {sum(x) / len(x)}")

# Median calculation
sorted_x = sorted(x)
n = len(sorted_x)
if n % 2 == 0:
    median = (sorted_x[n//2 - 1] + sorted_x[n//2]) / 2
else:
    median = sorted_x[n//2]
print(f"Median: {median}")

# Mode - most frequent value
from collections import Counter
counts = Counter(x)
mode = counts.most_common(1)[0][0]
print(f"Mode: {mode} (appears {counts.most_common(1)[0][1]} times)")

Data: [2, 4, 4, 4, 5, 5, 7, 9]
Mean: 5.0
Median: 4.5
Mode: 4 (appears 3 times)


# Outlier Detection

Real data often has weird values that can mess up your analysis. Need to find and understand these outliers.

## Patient Data Analysis

We'll analyze a patient dataset to calculate descriptive statistics and identify outliers using the Interquartile Range (IQR) method. This approach helps detect unusual observations that might indicate data entry errors or genuinely exceptional cases.

In [72]:
# Generate synthetic patient dataset with extreme outliers
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)

# Create patient data with realistic distributions
normal_data = pd.DataFrame({
    'age': np.random.normal(35, 8, size=90),       # age in years
    'weight': np.random.normal(70, 10, size=90),   # weight in kg
    'height': np.random.normal(170, 7, size=90)    # height in cm
})

# Add EXTREME outliers to clearly demonstrate impact on central tendency
extreme_outliers = pd.DataFrame({
    'age': [150, 2, 125, 1, 140, 0, 130, 180, 160, 200, 175],        # impossible ages
    'weight': [300, 15, 350, 10, 400, 8, 320, 280, 450, 500, 380],   # extreme weights
    'height': [250, 100, 280, 90, 300, 85, 270, 260, 310, 320, 290]  # impossible heights
})

# Combine normal data with extreme outliers
df = pd.concat([normal_data, extreme_outliers], ignore_index=True)

print("Dataset info:")
df.info()
print(f"\nTotal records: {len(df)} ({len(normal_data)} normal + {len(extreme_outliers)} extreme outliers)")
print(f"\nSample of extreme values:")
print("Age range:", df['age'].min(), "to", df['age'].max())
print("Weight range:", df['weight'].min(), "to", df['weight'].max())  
print("Height range:", df['height'].min(), "to", df['height'].max())

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     101 non-null    float64
 1   weight  101 non-null    float64
 2   height  101 non-null    float64
dtypes: float64(3)
memory usage: 2.5 KB

Total records: 101 (90 normal + 11 extreme outliers)

Sample of extreme values:
Age range: 0.0 to 200.0
Weight range: 8.0 to 500.0
Height range: 85.0 to 320.0


## Central Tendency Measures

Calculate mean, median, and mode for each variable to understand the typical values and see how outliers affect these measures.

In [73]:
# Calculate descriptive statistics and show dramatic impact of extreme outliers
print("DESCRIPTIVE STATISTICS: IMPACT OF EXTREME OUTLIERS")
print("=" * 55)

means = df.mean()
medians = df.median()
means_clean = normal_data.mean()
medians_clean = normal_data.median()

print("WITH EXTREME OUTLIERS:")
for col in df.columns:
    print(f"  {col}: mean={means[col]:.1f}, median={medians[col]:.1f}")

print("\nWITHOUT OUTLIERS (normal data only):")
for col in normal_data.columns:
    print(f"  {col}: mean={means_clean[col]:.1f}, median={medians_clean[col]:.1f}")

print("\n" + "="*55)
print("IMPACT OF EXTREME OUTLIERS:")

for col in df.columns:
    mean_diff = means[col] - means_clean[col]
    median_diff = medians[col] - medians_clean[col]
    mean_pct = (mean_diff / means_clean[col]) * 100
    median_pct = (median_diff / medians_clean[col]) * 100 if medians_clean[col] != 0 else 0
    
    print(f"\n{col.upper()}:")
    print(f"  Mean: {means_clean[col]:.1f} → {means[col]:.1f} (change: {mean_diff:+.1f}, {mean_pct:+.1f}%)")
    print(f"  Median: {medians_clean[col]:.1f} → {medians[col]:.1f} (change: {median_diff:+.1f}, {median_pct:+.1f}%)")

print("\n" + "="*55)
print("KEY INSIGHT:")
print("Mean is heavily influenced by extreme values (large % changes)")
print("Median remains robust against outliers (small % changes)")

# Enhanced summary table
summary = pd.DataFrame({
    'Mean_Clean': means_clean,
    'Mean_Outliers': means,
    'Mean_Change': means - means_clean,
    'Median_Clean': medians_clean,
    'Median_Outliers': medians,
    'Median_Change': medians - medians_clean
}).round(1)

print("\nDETAILED COMPARISON:")
print(summary)

DESCRIPTIVE STATISTICS: IMPACT OF EXTREME OUTLIERS
WITH EXTREME OUTLIERS:
  age: mean=43.0, median=34.7
  weight: mean=92.7, median=71.7
  height: mean=176.7, median=170.9

WITHOUT OUTLIERS (normal data only):
  age: mean=34.2, median=34.0
  weight: mean=70.5, median=70.8
  height: mean=169.9, median=170.4

IMPACT OF EXTREME OUTLIERS:

AGE:
  Mean: 34.2 → 43.0 (change: +8.8, +25.7%)
  Median: 34.0 → 34.7 (change: +0.7, +2.1%)

WEIGHT:
  Mean: 70.5 → 92.7 (change: +22.1, +31.4%)
  Median: 70.8 → 71.7 (change: +0.9, +1.3%)

HEIGHT:
  Mean: 169.9 → 176.7 (change: +6.8, +4.0%)
  Median: 170.4 → 170.9 (change: +0.5, +0.3%)

KEY INSIGHT:
Mean is heavily influenced by extreme values (large % changes)
Median remains robust against outliers (small % changes)

DETAILED COMPARISON:
        Mean_Clean  Mean_Outliers  Mean_Change  Median_Clean  Median_Outliers  \
age           34.2           43.0          8.8          34.0             34.7   
weight        70.5           92.7         22.1          

## IQR Method for Outlier Detection

The Interquartile Range method is robust for identifying outliers since it's based on quartiles rather than mean/standard deviation.

The IQR is the difference between the 75th percentile (Q3) and 25th percentile (Q1):

$$\text{IQR} = Q_3 - Q_1$$

Outliers are defined as observations falling outside these bounds:
- Lower bound: $Q_1 - 1.5 \times \text{IQR}$
- Upper bound: $Q_3 + 1.5 \times \text{IQR}$

Any value below the lower bound or above the upper bound is flagged as an outlier.

The factor 1.5 is conventional and works well for most datasets - it captures roughly the most extreme 0.7% of normally distributed data.

In [74]:
# Apply IQR method
print("OUTLIER DETECTION USING IQR")
print("-" * 35)

Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

print("Quartiles:")
print(f"Q1 (25th percentile): {Q1.round(2).to_dict()}")
print(f"Q3 (75th percentile): {Q3.round(2).to_dict()}")
print(f"IQR: {IQR.round(2).to_dict()}")

# Calculate bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nOutlier boundaries:")
print(f"Lower bounds: {lower_bound.round(2).to_dict()}")
print(f"Upper bounds: {upper_bound.round(2).to_dict()}")

# Identify outliers
outlier_mask = (df < lower_bound) | (df > upper_bound)

print(f"\nOutlier count by variable:")
for col in df.columns:
    n_outliers = outlier_mask[col].sum()
    print(f"  {col}: {n_outliers} outlier(s)")

# Show outlier cases
outlier_rows = df[outlier_mask.any(axis=1)]
print(f"\nPatients with outliers ({len(outlier_rows)} cases):")
print(outlier_rows.round(2))

OUTLIER DETECTION USING IQR
-----------------------------------
Quartiles:
Q1 (25th percentile): {'age': 30.19, 'weight': 62.86, 'height': 164.59}
Q3 (75th percentile): {'age': 41.14, 'weight': 77.91, 'height': 175.42}
IQR: {'age': 10.95, 'weight': 15.05, 'height': 10.83}

Outlier boundaries:
Lower bounds: {'age': 13.76, 'weight': 40.28, 'height': 148.34}
Upper bounds: {'age': 57.57, 'weight': 100.49, 'height': 191.67}

Outlier count by variable:
  age: 11 outlier(s)
  weight: 11 outlier(s)
  height: 13 outlier(s)

Patients with outliers (13 cases):
        age  weight  height
29    32.67   77.52  196.97
82    46.82   69.23  147.31
90   150.00  300.00  250.00
91     2.00   15.00  100.00
92   125.00  350.00  280.00
93     1.00   10.00   90.00
94   140.00  400.00  300.00
95     0.00    8.00   85.00
96   130.00  320.00  270.00
97   180.00  280.00  260.00
98   160.00  450.00  310.00
99   200.00  500.00  320.00
100  175.00  380.00  290.00


# Probability Theory

Basic concepts for understanding uncertainty and random events.

## Key Definitions

**Sample Space (S)**: Set of all possible outcomes
$$S = \{ \omega_1, \omega_2, \dots, \omega_n \}$$

**Event (A)**: Any subset of the sample space
$$A \subseteq S$$

**Probability**: Numerical measure of likelihood, ranging from 0 to 1
$$P(A) = \frac{\text{favorable outcomes}}{\text{total outcomes}}$$

**Conditional Probability**: Probability of A given that B occurred
$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} \quad \text{when } P(B) > 0$$

**Bayes' Theorem**: Updates probabilities with new evidence
$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Where:
- P(A): prior probability
- P(B|A): likelihood  
- P(B): marginal probability
- P(A|B): posterior probability

## Example: Urn with Colored Balls

An urn contains:
- 4 red balls
- 3 blue balls  
- 2 green balls

We draw one ball at random.

### Problem 1: Basic Probability

What's the probability of drawing a blue ball?

**Solution:**
- Sample space: S = {4 red, 3 blue, 2 green} 
- Total outcomes: |S| = 9
- Favorable outcomes: 3 blue balls
- P(blue) = 3/9 = 1/3 ≈ 0.333

In [75]:
# Basic probability calculation
red_balls = 4
blue_balls = 3  
green_balls = 2
total_balls = red_balls + blue_balls + green_balls

P_blue = blue_balls / total_balls

print("Urn contents:")
print(f"  Red: {red_balls}, Blue: {blue_balls}, Green: {green_balls}")
print(f"  Total: {total_balls}")
print(f"\nP(Blue) = {blue_balls}/{total_balls} = {P_blue:.3f}")

# Verify probabilities sum to 1
P_red = red_balls / total_balls
P_green = green_balls / total_balls
print(f"\nVerification: {P_red:.3f} + {P_blue:.3f} + {P_green:.3f} = {P_red + P_blue + P_green:.3f}")

Urn contents:
  Red: 4, Blue: 3, Green: 2
  Total: 9

P(Blue) = 3/9 = 0.333

Verification: 0.444 + 0.333 + 0.222 = 1.000


## Conditional Probability

### Problem 2: Given NOT red, what's P(blue)?

We draw a ball and know it's not red. What's the probability it's blue?

**Solution:**

Let A = "blue ball", B = "not red ball"

Since blue ⊆ not red: P(A ∩ B) = P(A) = 3/9

P(B) = P(blue or green) = 3/9 + 2/9 = 5/9

Using conditional probability:
$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} = \frac{3/9}{5/9} = \frac{3}{5} = 0.6$$

The additional information (not red) increases our confidence from 33% to 60%.

In [76]:
# Conditional probability calculation
P_blue = blue_balls / total_balls
P_green = green_balls / total_balls  
P_not_red = P_blue + P_green

print("CONDITIONAL PROBABILITY")
print("-" * 25)
print("Given: ball is NOT red")
print("Find: P(blue | not red)")

print(f"\nP(blue) = {P_blue:.3f}")
print(f"P(not red) = P(blue) + P(green) = {P_blue:.3f} + {P_green:.3f} = {P_not_red:.3f}")

P_blue_given_not_red = P_blue / P_not_red

print(f"\nP(blue | not red) = P(blue) / P(not red) = {P_blue:.3f} / {P_not_red:.3f} = {P_blue_given_not_red:.3f}")

print(f"\nComparison:")
print(f"  Before: P(blue) = {P_blue:.1%}")
print(f"  After:  P(blue | not red) = {P_blue_given_not_red:.1%}")
print(f"  Information gain: +{(P_blue_given_not_red - P_blue)*100:.1f} percentage points")

CONDITIONAL PROBABILITY
-------------------------
Given: ball is NOT red
Find: P(blue | not red)

P(blue) = 0.333
P(not red) = P(blue) + P(green) = 0.333 + 0.222 = 0.556

P(blue | not red) = P(blue) / P(not red) = 0.333 / 0.556 = 0.600

Comparison:
  Before: P(blue) = 33.3%
  After:  P(blue | not red) = 60.0%
  Information gain: +26.7 percentage points


## Independence of Events

Two events A and B are independent when the outcome of one doesn't affect the other.

**Mathematical definition:**
$$P(A \cap B) = P(A) \times P(B)$$

**Equivalent conditions:**
- P(A|B) = P(A)  
- P(B|A) = P(B)

**Examples:**
- Independent: coin flips, die rolls, rain vs stock prices
- Dependent: height vs weight, study time vs grades

### Example: Two Coin Flips

Flip two fair coins. Events:
- A: first coin shows heads
- B: second coin shows heads

Are these independent?

In [77]:
# Two coin flip example
P_heads_1 = 0.5
P_heads_2 = 0.5

print("INDEPENDENCE: TWO COIN FLIPS")
print("-" * 32)
print("Sample space: {HH, HT, TH, TT}")

# For independent events: P(A ∩ B) = P(A) × P(B)
P_both_heads = P_heads_1 * P_heads_2

print(f"\nP(first = H) = {P_heads_1}")
print(f"P(second = H) = {P_heads_2}")
print(f"P(both = H) = {P_heads_1} × {P_heads_2} = {P_both_heads}")

# Verify by counting outcomes
print(f"\nVerification by counting:")
print(f"  Favorable outcomes: HH")
print(f"  Total outcomes: 4")
print(f"  P(both heads) = 1/4 = 0.25 ✓")

# Test independence
print(f"\nIndependence test:")
print(f"  P(second = H | first = H) = {P_heads_2} (unchanged)")
print(f"  P(second = H) = {P_heads_2}")
print(f"  Equal → Independent ✓")

INDEPENDENCE: TWO COIN FLIPS
--------------------------------
Sample space: {HH, HT, TH, TT}

P(first = H) = 0.5
P(second = H) = 0.5
P(both = H) = 0.5 × 0.5 = 0.25

Verification by counting:
  Favorable outcomes: HH
  Total outcomes: 4
  P(both heads) = 1/4 = 0.25 ✓

Independence test:
  P(second = H | first = H) = 0.5 (unchanged)
  P(second = H) = 0.5
  Equal → Independent ✓


# Bayes' Theorem

Shows how to update probabilities when you get new evidence. Fundamental for rational decision-making under uncertainty.

**Formula:**
$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

**Components:**
- P(A|B): posterior - what we want to find
- P(B|A): likelihood - how likely the evidence is given our hypothesis
- P(A): prior - our initial belief
- P(B): marginal probability - total probability of the evidence

Bayes' theorem combines prior knowledge with new evidence to give updated beliefs.

## Medical Diagnosis Example

A patient tests positive for a rare disease. What's the actual probability they have it?

**Setup:**
- Disease prevalence: 0.1% of population
- Test sensitivity: 95% (detects disease when present)
- Test specificity: 98% (correct when disease absent)
- Patient tests positive

**Question:** P(disease | positive test) = ?

**Solution:**

Define events: D = disease, T+ = positive test

Given probabilities:
- P(D) = 0.001 (disease prevalence)
- P(T+|D) = 0.95 (sensitivity)  
- P(T-|D^c) = 0.98 (specificity)
- P(T+|D^c) = 0.02 (false positive rate)

Calculate P(T+) using law of total probability:
P(T+) = P(T+|D)×P(D) + P(T+|D^c)×P(D^c)
P(T+) = 0.95×0.001 + 0.02×0.999 = 0.02093

Apply Bayes' theorem:
P(D|T+) = P(T+|D)×P(D) / P(T+) = (0.95×0.001) / 0.02093 = 0.0454

**Result:** Only 4.54% chance of having the disease despite positive test!

In [78]:
# Bayes' theorem calculation
print("BAYES' THEOREM: MEDICAL DIAGNOSIS")
print("-" * 38)

# Given information
P_disease = 0.001
P_no_disease = 0.999
P_pos_given_disease = 0.95      # sensitivity
P_neg_given_no_disease = 0.98   # specificity
P_pos_given_no_disease = 0.02   # false positive rate

print("Given:")
print(f"  Disease prevalence: {P_disease}")
print(f"  Test sensitivity: {P_pos_given_disease}")
print(f"  Test specificity: {P_neg_given_no_disease}")

# Calculate marginal probability P(Test+)
P_positive = (P_pos_given_disease * P_disease + 
              P_pos_given_no_disease * P_no_disease)

print(f"\nCalculate P(Test+):")
print(f"  P(Test+) = {P_pos_given_disease}×{P_disease} + {P_pos_given_no_disease}×{P_no_disease}")
print(f"  P(Test+) = {P_positive:.6f}")

# Apply Bayes' theorem
P_disease_given_positive = (P_pos_given_disease * P_disease) / P_positive

print(f"\nApply Bayes' theorem:")
print(f"  P(Disease|Test+) = ({P_pos_given_disease}×{P_disease}) / {P_positive:.6f}")
print(f"  P(Disease|Test+) = {P_disease_given_positive:.4f} = {P_disease_given_positive*100:.1f}%")

print(f"\nKey insight:")
print(f"  Despite 95% test accuracy, only {P_disease_given_positive*100:.1f}% chance of disease")
print(f"  Base rate matters! Most positives are false alarms from healthy population")

BAYES' THEOREM: MEDICAL DIAGNOSIS
--------------------------------------
Given:
  Disease prevalence: 0.001
  Test sensitivity: 0.95
  Test specificity: 0.98

Calculate P(Test+):
  P(Test+) = 0.95×0.001 + 0.02×0.999
  P(Test+) = 0.020930

Apply Bayes' theorem:
  P(Disease|Test+) = (0.95×0.001) / 0.020930
  P(Disease|Test+) = 0.0454 = 4.5%

Key insight:
  Despite 95% test accuracy, only 4.5% chance of disease
  Base rate matters! Most positives are false alarms from healthy population


## Why Such a Low Result?

Let's see what happens if we test 100,000 people:

**Population:**
- With disease: 100,000 × 0.001 = 100 people
- Without disease: 99,900 people

**Test results:**
- True positives: 100 × 0.95 = 95
- False positives: 99,900 × 0.02 = 1,998
- Total positives: 2,093

**Among the 2,093 positive tests:**
- Actually sick: 95
- Actually healthy: 1,998

Therefore: P(disease | positive) = 95/2,093 = 4.54%

The key insight: even with high test accuracy, false positives from the huge healthy population dominate the results.

In [79]:
# Demonstrate with concrete numbers
population = 100000

sick_people = int(population * P_disease)
healthy_people = population - sick_people

true_positives = int(sick_people * P_pos_given_disease)
false_positives = int(healthy_people * P_pos_given_no_disease)
total_positives = true_positives + false_positives

print("INTUITIVE EXPLANATION")
print("-" * 22)
print(f"Test {population:,} people:")

print(f"\nPopulation:")
print(f"  Sick: {sick_people}")
print(f"  Healthy: {healthy_people:,}")

print(f"\nTest results:")
print(f"  True positives: {true_positives}")
print(f"  False positives: {false_positives:,}")
print(f"  Total positives: {total_positives:,}")

actual_prob = true_positives / total_positives
print(f"\nAmong {total_positives:,} positive tests:")
print(f"  Actually sick: {true_positives} ({actual_prob:.1%})")
print(f"  Actually healthy: {false_positives:,} ({1-actual_prob:.1%})")

print(f"\nWhy so many false positives?")
print(f"  {healthy_people//sick_people:,}x more healthy people than sick")
print(f"  Even 2% error rate creates {false_positives:,} false alarms")
print(f"  This swamps the {true_positives} true cases")

INTUITIVE EXPLANATION
----------------------
Test 100,000 people:

Population:
  Sick: 100
  Healthy: 99,900

Test results:
  True positives: 95
  False positives: 1,998
  Total positives: 2,093

Among 2,093 positive tests:
  Actually sick: 95 (4.5%)
  Actually healthy: 1,998 (95.5%)

Why so many false positives?
  999x more healthy people than sick
  Even 2% error rate creates 1,998 false alarms
  This swamps the 95 true cases


## Applications

**Medical & Healthcare:**
- Disease screening and diagnosis
- Drug effectiveness studies
- Epidemic modeling

**Machine Learning:**
- Spam filters (Naive Bayes)
- Recommendation systems
- Natural language processing

**Business & Finance:**
- Credit scoring
- Fraud detection
- Market research and A/B testing

**Legal & Forensics:**
- DNA evidence evaluation
- Eyewitness reliability assessment

**Key Takeaways:**
- Base rates (priors) matter enormously
- Single tests/evidence rarely definitive  
- Always consider false positive rates
- Bayes helps overcome cognitive biases
- Framework for rational thinking under uncertainty